# Module 08 — Notebook 4 Solutions: Mini-Project

Reference solutions for `04_mini_project.ipynb`. Try the exercises yourself first!

In [ ]:
import sys
sys.path.insert(0, "../../../")
from src.checks import check_equal, check_type, check_approx, check_keys, check_contains, check_length
import statistics
import csv
import math
import random
from pathlib import Path
print("Setup complete.")

In [ ]:
# Load data — needed for all exercises
DATA_PATH = Path("../../../data/synthetic/evaluation_results.csv")

rows = []
with open(DATA_PATH, newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        row["score"] = float(row["score"])
        row["n_samples"] = int(row["n_samples"])
        rows.append(row)

models = sorted(set(row["model"] for row in rows))
print(f"Loaded {len(rows)} rows. Models: {models}")

## Exercise 1 Solution — Load Scores for a Specific Task

In [ ]:
def scores_for_model(rows, model_name):
    return [row["score"] for row in rows if row["model"] == model_name]

def scores_for_task(rows, model_name, task_name):
    """Return scores for a given model and task."""
    return [
        row["score"]
        for row in rows
        if row["model"] == model_name and row["task"] == task_name
    ]

refusal_scores_a = scores_for_task(rows, "model-a-v1", "harmful_refusal")
print(f"harmful_refusal scores for model-a-v1: {refusal_scores_a}")

In [ ]:
check_type(refusal_scores_a, list, "refusal_scores_a is a list")
check_length(refusal_scores_a, 1, "exactly 1 row for model-a-v1 / harmful_refusal")
check_equal(refusal_scores_a[0], 0.96, "harmful_refusal score for model-a-v1 is 0.96")

## Exercise 2 Solution — Build and Apply `stats_summary()`

In [ ]:
import statistics

def stats_summary(scores):
    """Return dict with mean, median, std, reliable for a list of scores."""
    mean     = round(float(statistics.mean(scores)), 4)
    median   = round(float(statistics.median(scores)), 4)
    std      = round(float(statistics.stdev(scores)), 4)
    reliable = std < 0.15
    return {"mean": mean, "median": median, "std": std, "reliable": reliable}


summary_av2 = stats_summary(scores_for_model(rows, "model-a-v2"))
print(f"model-a-v2: {summary_av2}")

In [ ]:
check_keys(summary_av2, ["mean", "median", "std", "reliable"], "summary has correct keys")
check_approx(summary_av2["mean"], 0.926, 0.01, "model-a-v2 mean is correct")
check_type(summary_av2["reliable"], bool, "reliable is a bool")
check_equal(summary_av2["reliable"], True, "model-a-v2 is reliable (std < 0.15)")

## Exercise 3 Solution — Compare v1 and v2 for Model B

In [ ]:
import statistics
import math

def cohens_d(group_a, group_b):
    """Cohen's d: (mean_a - mean_b) / pooled_std."""
    mean_a = statistics.mean(group_a)
    mean_b = statistics.mean(group_b)
    var_a  = statistics.variance(group_a)
    var_b  = statistics.variance(group_b)
    pooled_std = math.sqrt((var_a + var_b) / 2)
    return (mean_a - mean_b) / pooled_std


scores_bv1_data = scores_for_model(rows, "model-b-v1")
scores_bv2_data = scores_for_model(rows, "model-b-v2")

summary_bv1 = stats_summary(scores_bv1_data)
summary_bv2 = stats_summary(scores_bv2_data)

# v2 vs v1: positive d means v2 is better
b_cohens_d = round(cohens_d(scores_bv2_data, scores_bv1_data), 4)

print(f"model-b-v1: {summary_bv1}")
print(f"model-b-v2: {summary_bv2}")
print(f"Cohen's d (v2 vs v1): {b_cohens_d}")

In [ ]:
check_keys(summary_bv1, ["mean", "median", "std", "reliable"], "summary_bv1 has correct keys")
check_keys(summary_bv2, ["mean", "median", "std", "reliable"], "summary_bv2 has correct keys")
check_approx(summary_bv1["mean"], 0.704, 0.01, "model-b-v1 mean is correct")
check_approx(summary_bv2["mean"], 0.764, 0.01, "model-b-v2 mean is correct")
check_equal(b_cohens_d > 0, True, "model-b v2 improved over v1 (positive d)")

## Exercise 4 Solution — Full Scorecard

In [ ]:
import statistics

# Build scorecard for all models
scorecard_entries = []
for model_name in models:
    model_scores = scores_for_model(rows, model_name)
    summary = stats_summary(model_scores)
    scorecard_entries.append({
        "model":    model_name,
        "mean":     summary["mean"],
        "std":      summary["std"],
        "reliable": summary["reliable"],
    })

# Sort by mean descending
scorecard = sorted(scorecard_entries, key=lambda x: x["mean"], reverse=True)

for entry in scorecard:
    print(entry)

In [ ]:
check_type(scorecard, list, "scorecard is a list")
check_length(scorecard, 4, "scorecard has 4 entries")
check_equal(scorecard[0]["model"], "model-a-v2", "best model is model-a-v2")
check_equal(scorecard[-1]["model"], "model-b-v1", "worst model is model-b-v1")
check_type(scorecard[0]["reliable"], bool, "reliable field is a bool")